Import libraries

In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors

Load the dataset

In [2]:
# Load the dataset
file_path = 'mp_dataset_initial.xlsx'
mp_dataset_initial = pd.read_excel(file_path, engine='openpyxl')
mp_dataset_initial

,Formulation Index,Drug,Drug SMILES,Polymer Mw,Polymer Mn,Polymer Molecular Weight (unit not specified),PDI,LA/GA,Formulation Method,Initial Drug-to-Polymer Ratio,Particle Size,Drug Loading Capacity,Drug Encapsulation Efficiency,Solubility Enhancer Concentration,Time,Release,DOI
0,1,moxidectin,CC1CC(=CCC2CC(CC3(O2)CC(=NOC)C(C(O3)C(=CC(C)C)...,75.0,NaN,NaN,NaN,3.0,O/W,0.666667,47.723,35.41,88.30,0.5,0.000000,0.000000,10.3390/ijms241914729
1,1,moxidectin,CC1CC(=CCC2CC(CC3(O2)CC(=NOC)C(C(O3)C(=CC(C)C)...,75.0,NaN,NaN,NaN,3.0,O/W,0.666667,47.723,35.41,88.30,0.5,1.889764,0.062622,10.3390/ijms241914729
2,1,moxidectin,CC1CC(=CCC2CC(CC3(O2)CC(=NOC)C(C(O3)C(=CC(C)C)...,75.0,NaN,NaN,NaN,3.0,O/W,0.666667,47.723,35.41,88.30,0.5,7.181102,0.056751,10.3390/ijms241914729
3,1,moxidectin,CC1CC(=CCC2CC(CC3(O2)CC(=NOC)C(C(O3)C(=CC(C)C)...,75.0,NaN,NaN,NaN,3.0,O/W,0.666667,47.723,35.41,88.30,0.5,13.984252,0.058708,10.3390/ijms241914729
4,1,moxidectin,CC1CC(=CCC2CC(CC3(O2)CC(=NOC)C(C(O3)C(=CC(C)C)...,75.0,NaN,NaN,NaN,3.0,O/W,0.666667,47.723,35.41,88.30,0.5,27.968504,0.058708,10.3390/ijms241914729
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4908,321,triptolide,O=C1OCC2=C1CCC3(C)C2CC4OC45C(O)C6(OC6C7OC753)C...,NaN,NaN,20.0,NaN,3.0,O/W,0.112347,42.360,7.96,80.16,0.0,3.078327,0.210131,10.1371/journal. pone.0292861
4909,321,triptolide,O=C1OCC2=C1CCC3(C)C2CC4OC45C(O)C6(OC6C7OC753)C...,NaN,NaN,20.0,NaN,3.0,O/W,0.112347,42.360,7.96,80.16,0.0,7.068903,0.337711,10.1371/journal. pone.0292861
4910,321,triptolide,O=C1OCC2=C1CCC3(C)C2CC4OC45C(O)C6(OC6C7OC753)C...,NaN,NaN,20.0,NaN,3.0,O/W,0.112347,42.360,7.96,80.16,0.0,14.089168,0.613508,10.1371/journal. pone.0292861
4911,321,triptolide,O=C1OCC2=C1CCC3(C)C2CC4OC45C(O)C6(OC6C7OC753)C...,NaN,NaN,20.0,NaN,3.0,O/W,0.112347,42.360,7.96,80.16,0.0,21.110465,0.911820,10.1371/journal. pone.0292861


In [3]:
experiment_index_unique = mp_dataset_initial['Formulation Index'].nunique()
drug_unique = mp_dataset_initial['Drug'].nunique()
doi_unique = mp_dataset_initial['DOI'].nunique()
timepoints = mp_dataset_initial.shape[0]

# Print the results
print(f"Number of Formulations: {experiment_index_unique}")
print(f"Number of Drugs: {drug_unique}")
print(f"Number of Papers: {doi_unique}")
print(f"Number of Datapoints : {timepoints}")

Number of Formulations: 321
Number of Drugs: 89
Number of Papers: 113
Number of Datapoints : 4913


In [4]:
mp_dataset_initial_formulation = mp_dataset_initial.drop_duplicates(subset=['Formulation Index'], keep='first')
mp_dataset_initial_formulation = mp_dataset_initial_formulation.drop(['Time', 'Release'], axis=1)

In [6]:
weight_based = mp_dataset_initial_formulation['Polymer Mw'].notnull().sum()
number_based = mp_dataset_initial_formulation['Polymer Mn'].notnull().sum()
MW = mp_dataset_initial_formulation['Polymer Molecular Weight (unit not specified)'].notnull().sum()
PDI = mp_dataset_initial_formulation['PDI'].notnull().sum()

# Print the results
print(f"Number of Formulations reporting weight-based polymer MW: {weight_based}, {round(weight_based/321,2)*100}%")
print(f"Number of Formulations reporting number-based polymer MW: {number_based}, {round(number_based/321,2)*100}%")
print(f"Number of Formulations not reporting MW unit: {MW}, {round(MW/321,2)*100}%")
print(f"Number of Formulations reporting polymer PDI: {PDI}, {round(PDI/321,2)*100}%")

Number of Formulations reporting weight-based polymer MW: 190, 59.0%
Number of Formulations reporting number-based polymer MW: 17, 5.0%
Number of Formulations not reporting MW unit: 127, 40.0%
Number of Formulations reporting polymer PDI: 13, 4.0%


In [7]:
mp_dataset_initial_formulation.to_excel('mp_dataset_initial_formulation.xlsx', index=False)

Generate drug features

In [8]:
mp_dataset_processed = mp_dataset_initial.copy()

# Calculate Drug_MW, Drug_TPSA, and Drug_LogP
def calculate_properties(smiles):
    mol = Chem.MolFromSmiles(smiles)
    mw = Descriptors.MolWt(mol)
    tpsa = Descriptors.TPSA(mol)
    logp = Descriptors.MolLogP(mol)
    return pd.Series([mw, tpsa, logp])

mp_dataset_processed[['Drug MW', 'Drug TPSA', 'Drug LogP']] = mp_dataset_processed['Drug SMILES'].apply(calculate_properties)

In [9]:
# Create the new column 'Polymer MW'
mp_dataset_processed['Polymer MW'] = np.where(
    mp_dataset_processed['Polymer Mw'].notnull(), mp_dataset_processed['Polymer Mw'],
    np.where(
        mp_dataset_processed['Polymer Mn'].notnull(), mp_dataset_processed['Polymer Mn'],
        mp_dataset_processed['Polymer Molecular Weight (unit not specified)']
    )
)

In [10]:
mp_dataset_processed[['Drug MW', 'Drug TPSA', 'Drug LogP']]

,Drug MW,Drug TPSA,Drug LogP
0,639.830,116.04,5.7289
1,639.830,116.04,5.7289
2,639.830,116.04,5.7289
3,639.830,116.04,5.7289
4,639.830,116.04,5.7289
...,...,...,...
4908,360.406,84.12,1.1031
4909,360.406,84.12,1.1031
4910,360.406,84.12,1.1031
4911,360.406,84.12,1.1031


In [11]:
mp_dataset_processed

,Formulation Index,Drug,Drug SMILES,Polymer Mw,Polymer Mn,Polymer Molecular Weight (unit not specified),PDI,LA/GA,Formulation Method,Initial Drug-to-Polymer Ratio,...,Drug Loading Capacity,Drug Encapsulation Efficiency,Solubility Enhancer Concentration,Time,Release,DOI,Drug MW,Drug TPSA,Drug LogP,Polymer MW
0,1,moxidectin,CC1CC(=CCC2CC(CC3(O2)CC(=NOC)C(C(O3)C(=CC(C)C)...,75.0,NaN,NaN,NaN,3.0,O/W,0.666667,...,35.41,88.30,0.5,0.000000,0.000000,10.3390/ijms241914729,639.830,116.04,5.7289,75.0
1,1,moxidectin,CC1CC(=CCC2CC(CC3(O2)CC(=NOC)C(C(O3)C(=CC(C)C)...,75.0,NaN,NaN,NaN,3.0,O/W,0.666667,...,35.41,88.30,0.5,1.889764,0.062622,10.3390/ijms241914729,639.830,116.04,5.7289,75.0
2,1,moxidectin,CC1CC(=CCC2CC(CC3(O2)CC(=NOC)C(C(O3)C(=CC(C)C)...,75.0,NaN,NaN,NaN,3.0,O/W,0.666667,...,35.41,88.30,0.5,7.181102,0.056751,10.3390/ijms241914729,639.830,116.04,5.7289,75.0
3,1,moxidectin,CC1CC(=CCC2CC(CC3(O2)CC(=NOC)C(C(O3)C(=CC(C)C)...,75.0,NaN,NaN,NaN,3.0,O/W,0.666667,...,35.41,88.30,0.5,13.984252,0.058708,10.3390/ijms241914729,639.830,116.04,5.7289,75.0
4,1,moxidectin,CC1CC(=CCC2CC(CC3(O2)CC(=NOC)C(C(O3)C(=CC(C)C)...,75.0,NaN,NaN,NaN,3.0,O/W,0.666667,...,35.41,88.30,0.5,27.968504,0.058708,10.3390/ijms241914729,639.830,116.04,5.7289,75.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4908,321,triptolide,O=C1OCC2=C1CCC3(C)C2CC4OC45C(O)C6(OC6C7OC753)C...,NaN,NaN,20.0,NaN,3.0,O/W,0.112347,...,7.96,80.16,0.0,3.078327,0.210131,10.1371/journal. pone.0292861,360.406,84.12,1.1031,20.0
4909,321,triptolide,O=C1OCC2=C1CCC3(C)C2CC4OC45C(O)C6(OC6C7OC753)C...,NaN,NaN,20.0,NaN,3.0,O/W,0.112347,...,7.96,80.16,0.0,7.068903,0.337711,10.1371/journal. pone.0292861,360.406,84.12,1.1031,20.0
4910,321,triptolide,O=C1OCC2=C1CCC3(C)C2CC4OC45C(O)C6(OC6C7OC753)C...,NaN,NaN,20.0,NaN,3.0,O/W,0.112347,...,7.96,80.16,0.0,14.089168,0.613508,10.1371/journal. pone.0292861,360.406,84.12,1.1031,20.0
4911,321,triptolide,O=C1OCC2=C1CCC3(C)C2CC4OC45C(O)C6(OC6C7OC753)C...,NaN,NaN,20.0,NaN,3.0,O/W,0.112347,...,7.96,80.16,0.0,21.110465,0.911820,10.1371/journal. pone.0292861,360.406,84.12,1.1031,20.0


In [13]:
col_to_drop = ['Drug', 'Drug SMILES', 'Formulation Method', 'DOI', 'Polymer Mw', 'Polymer Mn', 'Polymer Molecular Weight (unit not specified)', 'PDI']
mp_dataset_processed = mp_dataset_processed.drop(col_to_drop, axis = 1)

In [15]:
# Create the new order of columns
new_order = (['Formulation Index','Drug MW', 'Drug TPSA', 'Drug LogP', 'Polymer MW', 'LA/GA', 'Initial Drug-to-Polymer Ratio', 'Particle Size', 'Drug Loading Capacity', 'Drug Encapsulation Efficiency', 'Solubility Enhancer Concentration', 'Time', 'Release']
)

# Reorder the DataFrame
mp_dataset_processed = mp_dataset_processed[new_order]

In [16]:
mp_dataset_processed.to_excel('mp_dataset_processed.xlsx', index=False)